In [1]:
import torch
import torch.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from typing import Tuple, Optional, List
from torch.func import vmap, jacrev
import torch.nn.functional as F
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#加入score matching部分

In [2]:
class Sampleable(ABC):
    @abstractmethod
    def sample(self, num_samples: int) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        pass

class ConditionalProbabilityPath(nn.Module, ABC):
    def __init__(self, p_simple: Sampleable, p_data: Sampleable):
        super().__init__()
        self.p_simple = p_simple
        self.p_data = p_data

    def sample_marginal_path(self, t: torch.Tensor) -> torch.Tensor:
        num_samples = t.shape[0]
        z, _ = self.sample_conditioning_variable(num_samples)
        x = self.sample_conditional_path(z, t)
        return x

    @abstractmethod
    def sample_conditioning_variable(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
        pass

    @abstractmethod  
    def sample_conditional_path(self, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        pass



In [3]:
# 更改MNIST （时间序列建模）
class SimpleTrajectoryDataset(Sampleable):
    def __init__(self, seq_len=50, dim=2, num_classes=3):
        self.seq_len = seq_len
        self.dim = dim
        self.num_classes = num_classes
    
    def sample(self, num_samples: int) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        labels = torch.randint(0, self.num_classes, (num_samples,))
        trajectories = []
        
        for label in labels:
            label_val = label.item()
            # 时间序列：从t=0到t=seq_len-1
            t = np.linspace(0, 2*np.pi, self.seq_len)
            
            if label_val == 0:  # 圆形时间序列
                x = np.cos(t) * 0.5
                y = np.sin(t) * 0.5
                traj = np.stack([x, y], axis=-1)
            elif label_val == 1:  # 直线时间序列
                x = np.linspace(-0.5, 0.5, self.seq_len)
                y = x * 0.5
                traj = np.stack([x, y], axis=-1)
            else:  # 椭圆时间序列
                x = np.cos(t) * 0.7
                y = np.sin(t) * 0.3
                traj = np.stack([x, y], axis=-1)
            
            trajectories.append(traj)
        
        trajectories = torch.tensor(np.array(trajectories), dtype=torch.float32).to(device)
        return trajectories, labels.to(device)

class TrajectoryNoise(Sampleable):
    def __init__(self, seq_len=50, dim=2):
        self.seq_len = seq_len
        self.dim = dim
    
    def sample(self, num_samples: int) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # 生成随机噪声轨迹
        trajectories = torch.randn(num_samples, self.seq_len, self.dim).to(device)
        return trajectories, None

In [4]:
# class LinearAlpha(nn.Module):
#     def forward(self, t):
#         return t.squeeze()

# class LinearBeta(nn.Module):
#     def forward(self, t):
#         return 1 - t.squeeze()

In [5]:
#alpha beta scheduler

class Alpha(ABC):
    def __init__(self):
        # Check alpha_t(0) = 0
        assert torch.allclose(
            self(torch.zeros(1,1,1)), torch.zeros(1,1,1)
        )
        # Check alpha_1 = 1
        assert torch.allclose(
            self(torch.ones(1,1,1)), torch.ones(1,1,1)
        )
        
    @abstractmethod
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 0.0, self(1.0) = 1.0.
        Args:
            - t: time (num_samples, 1, 1)
        Returns:
            - alpha_t (num_samples, 1, 1)
        """ 
        pass

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: time (num_samples, 1, 1)
        Returns:
            - d/dt alpha_t (num_samples, 1, 1)
        """ 
        t = t.unsqueeze(1)
        dt = vmap(jacrev(self))(t)
        return dt.view(-1, 1, 1)
    
class Beta(ABC):
    def __init__(self):
        # Check beta_0 = 1
        assert torch.allclose(
            self(torch.zeros(1,1,1)), torch.ones(1,1,1)
        )
        # Check beta_1 = 0
        assert torch.allclose(
            self(torch.ones(1,1,1)), torch.zeros(1,1,1)
        )
        
    @abstractmethod
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 1.0, self(1.0) = 0.0.
        Args:
            - t: time (num_samples, 1, 1)
        Returns:
            - beta_t (num_samples, 1, 1)
        """ 
        pass 

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt beta_t.
        Args:
            - t: time (num_samples, 1, 1)
        Returns:
            - d/dt beta_t (num_samples, 1, 1)
        """ 
        t = t.unsqueeze(1)
        dt = vmap(jacrev(self))(t)
        return dt.view(-1, 1, 1)

class LinearAlpha(Alpha):
    """
    Implements alpha_t = t
    """
    
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: time (num_samples, 1, 1)
        Returns:
            - alpha_t (num_samples, 1, 1)
        """ 
        return t
    
    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: time (num_samples, 1, 1)
        Returns:
            - d/dt alpha_t (num_samples, 1, 1)
        """ 
        return torch.ones_like(t)
        
class LinearBeta(Beta):
    """
    Implements beta_t = 1-t
    """
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: time (num_samples, 1)
        Returns:
            - beta_t (num_samples, 1)
        """ 
        return 1-t
        
    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: time (num_samples, 1, 1)
        Returns:
            - d/dt alpha_t (num_samples, 1, 1)
        """ 
        return - torch.ones_like(t)

In [6]:

class FourierEncoder(nn.Module):
    """傅里叶特征编码器，用于编码时间步"""
    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.embed_dim = embed_dim
        # 初始化频率
        self.frequencies = nn.Parameter(
            torch.randn(embed_dim // 2) * 2.0,
            requires_grad=False
        )
    
    def forward(self, t):
        """
        Args:
            t: shape (batch, 1, 1) 时间步
        Returns:
            shape (batch, embed_dim) 傅里叶编码
        """
        t = t.squeeze(-1).squeeze(-1)  # (batch,)
        # 计算 sin 和 cos 编码
        angles = t.unsqueeze(-1) * self.frequencies.unsqueeze(0) * 2 * 3.14159
        encoding = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return encoding  # (batch, embed_dim)


class SimpleTrajectoryUNet(nn.Module):
    def __init__(self, 
                 seq_len: int = 50,
                 input_dim: int = 2,  # 轨迹坐标维度(x,y)
                 hidden_dims: List[int] = [64, 128],
                 t_embed_dim: int = 64,
                 y_embed_dim: int = 32,
                 num_classes: int = 3):
        super().__init__()
        
        # 输入投影：(seq_len, input_dim) -> (seq_len, hidden_dim)
        self.input_proj = nn.Linear(input_dim, hidden_dims[0])
        
        # 扩散时间步编码器
        self.time_embedder = FourierEncoder(t_embed_dim)
        
        # 轨迹类别编码器
        self.y_embedder = nn.Embedding(num_classes, y_embed_dim)
        
        # 将时间和类别信息投影到隐藏维度
        self.time_proj = nn.Linear(t_embed_dim, hidden_dims[0])
        self.y_proj = nn.Linear(y_embed_dim, hidden_dims[0])

        # 编码器：用Conv1d处理时间序列特征
        self.encoder_convs = nn.ModuleList()
        self.downsample_layers = nn.ModuleList()
        
        for i in range(len(hidden_dims) - 1):
            # Conv1d卷积块：在时间维度上提取局部模式
            self.encoder_convs.append(
                nn.Sequential(
                    nn.Conv1d(hidden_dims[i], hidden_dims[i], kernel_size=3, padding=1),
                    nn.SiLU(),
                    nn.Conv1d(hidden_dims[i], hidden_dims[i], kernel_size=3, padding=1),
                    nn.SiLU()
                )
            )
            # 下采样：减少时间分辨率
            self.downsample_layers.append(
                nn.Conv1d(hidden_dims[i], hidden_dims[i+1], kernel_size=3, stride=2, padding=1)
            )
        
        # 中间层：最深层的特征处理
        self.mid_conv = nn.Sequential(
            nn.Conv1d(hidden_dims[-1], hidden_dims[-1], kernel_size=3, padding=1),
            nn.SiLU(),
            nn.Conv1d(hidden_dims[-1], hidden_dims[-1], kernel_size=3, padding=1),
            nn.SiLU()
        )
        
        # 解码器：恢复时间分辨率
        self.upsample_layers = nn.ModuleList()
        self.decoder_convs = nn.ModuleList()
        
        for i in range(len(hidden_dims) - 1, 0, -1):
            # 上采样：增加时间分辨率
            self.upsample_layers.append(
                nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='linear', align_corners=False),
                    nn.Conv1d(hidden_dims[i], hidden_dims[i-1], kernel_size=3, padding=1)
                )
            )
            # Conv1d解码块
            self.decoder_convs.append(
                nn.Sequential(
                    nn.Conv1d(hidden_dims[i-1], hidden_dims[i-1], kernel_size=3, padding=1),
                    nn.SiLU(),
                    nn.Conv1d(hidden_dims[i-1], hidden_dims[i-1], kernel_size=3, padding=1),
                    nn.SiLU()
                )
            )
        
        # 输出投影：(seq_len, hidden_dim) -> (seq_len, input_dim)
        self.out_v = nn.Conv1d(hidden_dims[0], input_dim, kernel_size=1)  # 速度头
        self.out_s = nn.Conv1d(hidden_dims[0], input_dim, kernel_size=1)  # score头

    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (bs, seq_len, input_dim) 时间序列轨迹
            t: (bs, 1, 1) 扩散时间步
            y: (bs,) 轨迹类别标签
        Returns:
            v_pred: (bs, seq_len, input_dim) 预测的速度场
            s_pred: (bs, seq_len, input_dim) 预测的分数场
        """
        bs, seq_len, input_dim = x.shape

        # 处理扩散时间步维度
        if t.dim() == 4:  # (bs, 1, 1, 1) -> (bs, 1, 1)
            t = t.squeeze(-1)
        
        # 编码扩散时间步和轨迹类别
        t_embed = self.time_embedder(t)  # (bs, t_embed_dim)
        y_embed = self.y_embedder(y)     # (bs, y_embed_dim)
        
        # 投影到隐藏维度
        t_proj = self.time_proj(t_embed).unsqueeze(-1)  # (bs, hidden_dim, 1)
        y_proj = self.y_proj(y_embed).unsqueeze(-1)     # (bs, hidden_dim, 1)
        
        # 输入投影
        x = self.input_proj(x)  # (bs, seq_len, hidden_dim)
        x = x.transpose(1, 2)   # (bs, hidden_dim, seq_len) 为Conv1d做准备
        
        # 添加时间和类别信息到所有时间步
        x = x + t_proj + y_proj
        
        # U-Net编码器
        skip_connections = []
        for conv, downsample in zip(self.encoder_convs, self.downsample_layers):
            x = conv(x)  # 在时间维度上卷积
            skip_connections.append(x)  # 保存跳跃连接
            x = downsample(x)  # 时间下采样

        # 中间层处理
        x = self.mid_conv(x)

        # U-Net解码器
        for upsample, conv in zip(self.upsample_layers, self.decoder_convs):
            x = upsample(x)  # 时间上采样
            skip = skip_connections.pop()  # 取出跳跃连接
            x = x + skip  # 残差连接
            x = conv(x)  # 在时间维度上卷积

        # 输出处理 - 直接从 (bs, hidden_dim, seq_len) 输出
        # x 现在是 (bs, hidden_dims[0], seq_len)
        v_pred = self.out_v(x)            # (bs, input_dim, seq_len)
        s_pred = self.out_s(x)            # (bs, input_dim, seq_len)
        
        # 转换为 (bs, seq_len, input_dim)
        v_pred = v_pred.permute(0, 2, 1)  # (bs, seq_len, input_dim)
        s_pred = s_pred.permute(0, 2, 1)  # (bs, seq_len, input_dim)
        
        return v_pred, s_pred

In [7]:
def plot_trajectories(trajectories, labels, title="Trajectories"):
    """可视化轨迹"""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    class_names = ['圆形', '直线', '椭圆']
    colors = ['red', 'blue', 'green']
    
    # 总览
    for class_idx in range(3):
        mask = labels == class_idx
        if mask.sum() > 0:
            class_trajs = trajectories[mask]
            for traj in class_trajs:
                axes[0].plot(traj[:, 0], traj[:, 1], color=colors[class_idx], alpha=0.6, linewidth=2)
    axes[0].set_title(f'{title} - 总览')
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.3)
    
    # 分类别显示
    for class_idx in range(3):
        mask = labels == class_idx
        if mask.sum() > 0:
            class_trajs = trajectories[mask]
            for traj in class_trajs:
                axes[class_idx + 1].plot(traj[:, 0], traj[:, 1], color=colors[class_idx], alpha=0.7, linewidth=2)
        axes[class_idx + 1].set_title(f'{class_names[class_idx]} (数量: {mask.sum()})')
        axes[class_idx + 1].set_aspect('equal')
        axes[class_idx + 1].grid(True, alpha=0.3)
        axes[class_idx + 1].set_xlim(-1, 1)
        axes[class_idx + 1].set_ylim(-0.6, 0.6)
    
    plt.tight_layout()
    plt.show()


def train_diffusion_model(lambda_fm=1.0, lambda_dsm=0.5, lambda_bridge=0.3):
    """训练扩散模型"""
    
    # 创建数据集
    trajectory_data = SimpleTrajectoryDataset(seq_len=50, num_classes=3)
    trajectory_noise = TrajectoryNoise(seq_len=50, dim=2)
    
    # 创建扩散系数
    alpha = LinearAlpha()
    beta = LinearBeta()
    
    # 创建条件概率路径
    cond_path = TrajectoryConditionalProbabilityPath(trajectory_noise, trajectory_data, alpha, beta)
    
    # 可视化真实数据
    print("展示真实轨迹时间序列...")
    real_trajs, real_labels = trajectory_data.sample(12)
    plot_trajectories(real_trajs.cpu(), real_labels.cpu(), "真实轨迹时间序列")
    
    # 创建U-Net模型
    unet = SimpleTrajectoryUNet(num_classes=3).to(device)
    print(f"模型参数数量: {sum(p.numel() for p in unet.parameters()):,}")
    
    # 优化器
    optimizer = torch.optim.Adam(unet.parameters(), lr=1e-3)
    
    # 训练参数
    num_epochs = 1000
    batch_size = 32
    losses = []
    losses_fm = []
    losses_dsm = []
    losses_bridge = []
    
    pbar = tqdm(range(num_epochs), desc="训练进度")
    
    for epoch in pbar:
        # 随机扩散时间步 (避免 t=0 导致除零)
        t = torch.rand(batch_size, 1, 1).to(device) * 0.999 + 0.001  # 范围 [0.001, 1.0]
        
        # 使用条件概率路径采样
        z, y = cond_path.sample_conditioning_variable(batch_size)  # 真实轨迹和标签
        
        # ====== 直线路径 (Flow Matching) ======
        x_t_line = cond_path.sample_conditional_path(z, t)
        target_v = cond_path.conditional_vector_field(x_t_line, z, t)
        v_pred_line, _ = unet(x_t_line, t, y)
        loss_fm = F.mse_loss(v_pred_line, target_v)
        
        # ====== 高斯路径 (DSM + Bridge) ======
        x_t_gauss = cond_path.sample_gaussian_path(z, t)
        v_pred_gauss, s_pred_gauss = unet(x_t_gauss, t, y)
        
        # Score Matching 损失
        alpha_t = cond_path.alpha(t).view(-1, 1, 1)
        sigma_t = cond_path.beta(t).view(-1, 1, 1)
        score_target = -(x_t_gauss - alpha_t * z) / (sigma_t**2 + 1e-8)
        loss_dsm = F.mse_loss(s_pred_gauss, score_target)
        
        # Bridge 损失
        alpha_dt = cond_path.alpha.dt(t).view(-1, 1, 1)
        sigma_dt = cond_path.beta.dt(t).view(-1, 1, 1)
        coef_x = alpha_dt / (alpha_t + 1e-8)
        coef_s = (alpha_dt * (sigma_t**2) / (alpha_t + 1e-8) - sigma_dt * sigma_t)
        u_from_s = coef_x * x_t_gauss + coef_s * s_pred_gauss
        loss_bridge = F.mse_loss(v_pred_gauss, u_from_s.detach())
        
        # ====== 合并损失并反传 ======
        loss = lambda_fm * loss_fm + lambda_dsm * loss_dsm + lambda_bridge * loss_bridge
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 记录损失
        losses.append(loss.item())
        losses_fm.append(loss_fm.item())
        losses_dsm.append(loss_dsm.item())
        losses_bridge.append(loss_bridge.item())
        
        pbar.set_postfix({
            'Total': f'{loss.item():.6f}',
            'FM': f'{loss_fm.item():.6f}',
            'DSM': f'{loss_dsm.item():.6f}',
            'Bridge': f'{loss_bridge.item():.6f}'
        })
    
    # 训练损失曲线
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(losses, label='Total Loss')
    plt.title('总损失曲线')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(losses_fm, label='FM Loss', alpha=0.7)
    plt.plot(losses_dsm, label='DSM Loss', alpha=0.7)
    plt.plot(losses_bridge, label='Bridge Loss', alpha=0.7)
    plt.title('分项损失曲线')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    return unet, cond_path, losses


def generate_trajectories(unet, cond_path, num_samples=12, num_steps=50):
    """使用 Flow Matching 生成轨迹"""
    print("生成新的轨迹时间序列...")
    unet.eval()
    
    with torch.no_grad():
        # 从噪声开始
        x = torch.randn(num_samples, 50, 2).to(device)
        
        # 指定生成的轨迹类别
        labels = torch.arange(3).repeat(num_samples // 3 + 1)[:num_samples].to(device)
        
        # ODE 求解：使用 Euler 方法积分速度场
        dt = 1.0 / num_steps
        
        for i in range(num_steps):
            t_val = i / num_steps  # 从 0 到 1
            t = torch.full((num_samples, 1, 1), t_val).to(device)
            
            # U-Net预测当前时刻的速度场
            v_t, _ = unet(x, t, labels)  # 修改：接收两个输出
            
            # Euler 积分：x_{t+dt} = x_t + v_t * dt
            x = x + v_t * dt
        
        return x.cpu(), labels.cpu()


def main():
    """主程序"""
    
    # 训练模型 (可以调整损失权重)
    trained_unet, cond_path, losses = train_diffusion_model(
        lambda_fm=1.0,
        lambda_dsm=0.5,
        lambda_bridge=0.3
    )
    
    # 生成轨迹
    generated_trajs, gen_labels = generate_trajectories(trained_unet, cond_path)
    
    # 可视化生成结果
    plot_trajectories(generated_trajs, gen_labels, "生成的轨迹时间序列")
    
    # 对比真实和生成轨迹
    real_trajs, real_labels = cond_path.p_data.sample(12)
    
    fig, axes = plt.subplots(2, 1, figsize=(20, 10))
    
    class_names = ['圆形', '直线', '椭圆']
    colors = ['red', 'blue', 'green']
    
    # 真实轨迹
    for class_idx in range(3):
        mask = real_labels.cpu() == class_idx
        if mask.sum() > 0:
            class_trajs = real_trajs.cpu()[mask]
            for traj in class_trajs:
                axes[0].plot(traj[:, 0], traj[:, 1], color=colors[class_idx], 
                           alpha=0.6, linewidth=2)
    axes[0].set_title('真实轨迹时间序列', fontsize=14)
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.3)
    
    # 生成轨迹
    for class_idx in range(3):
        mask = gen_labels == class_idx
        if mask.sum() > 0:
            class_trajs = generated_trajs[mask]
            for traj in class_trajs:
                axes[1].plot(traj[:, 0], traj[:, 1], color=colors[class_idx], 
                           alpha=0.6, linewidth=2)
    axes[1].set_title('生成的轨迹时间序列', fontsize=14)
    axes[1].set_aspect('equal')
    axes[1].grid(True, alpha=0.3)
    
    # 添加图例
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=colors[i], label=class_names[i]) for i in range(3)]
    axes[0].legend(handles=legend_elements, loc='upper right')
    axes[1].legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()

NameError: name 'TrajectoryConditionalProbabilityPath' is not defined